# Toy generation and amplitude fit

Toys are generated from the Belle 2018 model with TensorWaves' hit-and-miss
`IntensityDistributionGenerator`, and all free couplings are fitted with Minuit
(`tensorwaves.optimizer.Minuit2`) by minimising the unbinned negative log-likelihood
(`tensorwaves.estimator.UnbinnedNLL`). The $\rho(770)$ coupling is the reference,
fixed to 1.

In [ ]:
from operator import itemgetter

import matplotlib.pyplot as plt
import numpy as np
from tensorwaves.data import IntensityDistributionGenerator
from tensorwaves.data.rng import NumpyUniformRNG
from tensorwaves.estimator import UnbinnedNLL, create_cached_function
from tensorwaves.function.sympy import create_parametrized_function
from tensorwaves.optimizer import Minuit2

from aman_examples.d02kspipi import formulate_model
from aman_examples.phasespace import DalitzPlotGenerator

In [ ]:
model = formulate_model()
expression = model.intensity.doit()
intensity = create_parametrized_function(
    expression, model.parameter_defaults, backend="jax"
)
masses = [model.parameter_defaults[model.masses[k]] for k in ("pi", "KS", "pi", "D")]
phsp_generator = DalitzPlotGenerator(masses)
rng = NumpyUniformRNG(seed=1)
phsp = phsp_generator.generate(2_000_000, rng)
data_generator = IntensityDistributionGenerator(
    phsp_generator, intensity, bunch_size=200_000
)
data = data_generator.generate(50_000, rng)

## Fit

Only the couplings change during the fit. With `tensorwaves.estimator.create_cached_function`
everything else (lineshapes, spin factors, the K-matrix inverse) is computed once per
event: the data and phase-space samples are transformed into those cached
sub-expressions, and the likelihood only recombines them. This makes the fit about 300
times faster, so that a large normalisation sample can be used: with $5\times 10^5$
uniform events the normalisation integral is not precise enough for the narrow
$K^*(892)$ and $\omega(782)$ and the pulls are about 1.4 wide; with $2\times 10^6$
they are compatible with 1.

The fit starts from the Belle couplings shifted by a random 20%.

In [ ]:
free = {c: model.parameter_defaults[c] for c in model.free_couplings().values()}
free = {c: v for c, v in free.items() if v != 0}  # K*(1680)+ is not in the Belle model
cached_intensity, transformer = create_cached_function(
    expression, model.parameter_defaults, backend="jax", free_parameters=free
)
cached_data, cached_phsp = transformer(data), transformer(phsp)

shift = np.random.default_rng(2)
initial = {
    c.name: v * (1 + 0.2 * complex(*shift.normal(size=2))) for c, v in free.items()
}
estimator = UnbinnedNLL(cached_intensity, cached_data, cached_phsp, backend="jax")
result = Minuit2().optimize(estimator, initial)
print(
    f"valid: {result.minimum_valid}, {result.function_calls} calls, {result.execution_time:.1f} s"
)

In [ ]:
pulls = []
print(f"{'coupling':26s} {'fit':>22s} {'true':>18s}   pulls (re, im)")
for c, true in free.items():
    value, error = result.parameter_values[c.name], result.parameter_errors[c.name]
    pull = ((value - true).real / error.real, (value - true).imag / error.imag)
    pulls += pull
    print(f"{c.name:26s} {value:>22.4f} {true:>18.4f}   {pull[0]:+.2f} {pull[1]:+.2f}")
print(f"pulls: mean {np.mean(pulls):+.2f}, standard deviation {np.std(pulls):.2f}")

## Projections

In [ ]:
intensity.update_parameters(result.parameter_values)
weights = intensity(phsp)
msq = sum(m**2 for m in masses)


def m2_pipi(sample):
    return msq - sample["m2_kspim"] - sample["m2_kspip"]


fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), tight_layout=True)
for ax, (label, var) in zip(
    axes,
    [
        (r"$m^2(K^0_S\pi^-)$", itemgetter("m2_kspim")),
        (r"$m^2(K^0_S\pi^+)$", itemgetter("m2_kspip")),
        (r"$m^2(\pi^+\pi^-)$", m2_pipi),
    ],
    strict=True,
):
    counts, edges = np.histogram(var(data), bins=100)
    fitted, _ = np.histogram(var(phsp), bins=edges, weights=weights)
    centers = (edges[1:] + edges[:-1]) / 2
    ax.errorbar(
        centers, counts, yerr=np.sqrt(counts), fmt=".", color="black", label="toys"
    )
    ax.step(centers, fitted * counts.sum() / fitted.sum(), where="mid", label="fit")
    ax.set_xlabel(f"{label} [GeV$^2$]")
axes[0].legend()
plt.show()